# Blind SPRINT Test Notebook

Notebook này mô phỏng nhanh luồng Blind SPRINT theo đại số Schnorr trên trường hữu hạn nhỏ để kiểm tra tính đúng đắn.

In [ ]:
# Tham số demo (nhỏ để dễ nhìn kết quả)
p = 2**255 - 19  # mô phỏng modulo prime
G = 5            # giả lập generator ở dạng số

def H(*items):
    s = '|'.join(map(str, items)).encode()
    return int.from_bytes(__import__('hashlib').sha256(s).digest(), 'big') % p

def mod(x):
    return x % p

In [ ]:
# 1) Setup
import secrets
s = secrets.randbelow(p)
S = mod(s * G)

print('secret s =', s)
print('public S =', S)

In [ ]:
# 2) Preprocess 1 slot: R_bar = (r + delta)G
r = secrets.randbelow(p)
R = mod(r * G)
delta = H('delta', S, R)
R_bar = mod(R + delta * G)

# Bí mật slot thực chất là (r + delta)
slot_secret = mod(r + delta)
print('R_bar =', R_bar)

In [ ]:
# 3) Blind request
m = 'xin chao blind sprint'
ctx = 'ctx:test'
alpha = secrets.randbelow(p)
R_star = mod(R_bar + alpha * G)
e = H('challenge', S, R_star, m, ctx)

print('R* =', R_star)
print('e  =', e)

In [ ]:
# 4) Committee response + 5) Unblind
z_prime = mod(slot_secret + e * s)  # r + delta + e*s
z = mod(z_prime + alpha)
signature = (R_star, z)
print('signature =', signature)

In [ ]:
# 6) Verify Schnorr chuẩn: zG == R* + eS
left = mod(z * G)
right = mod(R_star + e * S)

print('left =', left)
print('right=', right)
print('VERIFY =', left == right)